# 01 · Recolección y calidad del dato

**Proyecto Hotel Booking Demand · Machine Learning · MUCSI 2026/27 · Universidad de Deusto**

> **TL;DR** — Antes de limpiar o modelar comprobamos tres cosas: que los datos **son los que creemos**,
> **qué no se puede usar** y si el lote **puede pasar** al notebook 02.
>
> - **Entra:** `data/raw/hotels.csv` (sin tocar).
> - **Sale:** 7 archivos en `outputs/01-data-collection-and-quality/` y una **decisión del lote**.
> - **No hace:** limpiar, explorar a fondo ni hacer gráficos. Eso es el notebook 02.

### El proyecto en una frase

Cuando entra una reserva, el hotel quiere saber **cuánto pagará por noche** (`adr`, regresión) y **si se
cancelará** (`is_canceled`, clasificación) para decidir depósitos y sobreventa.

### Mapa del notebook

Sigue el *Laboratorio 01* sección a sección. Cada sección empieza con su pregunta y su respuesta.

| § | Pregunta | Lab 01 |
|---|---|---|
| 1 | ¿Qué dataset necesitamos? | §2 |
| 2 | ¿Hemos recibido lo que dice la fuente? | §3 |
| 3 | ¿Qué es una fila? ¿Hay clave? ¿Hay duplicados? | §4 A |
| 4 | ¿Qué columnas no existen todavía cuando el modelo predice? | §4 B |
| 5 | ¿Cuánta calidad tienen los datos? | §5 |
| 6 | ¿A quién representan? | §6 |
| 7 | ¿Puede avanzar el lote? | §7 |
| 8 | ¿Qué garantizamos al notebook 02? | §8 |
| 9 | ¿Cómo se reproduce todo? | §9 y §11 |

## 0 · Preparar el entorno

Librerías, semilla, rutas y una función `check` que imprime ✅ o ⚠️ sin detener el notebook.

In [40]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Project root = the folder that contains requirements.txt (works from notebooks/ or from the root)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").exists())

RAW_PATH = ROOT / "content" / "data" / "raw" / "hotels.csv"
OUT_DIR = ROOT / "content" / "outputs" / "01-data-collection-and-quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def check(condition, ok, fail):
    """Print ✅ if the condition holds and ⚠️ otherwise."""
    print(f"✅ {ok}" if bool(condition) else f"⚠️ {fail}")


def sha256_file(path):
    """File fingerprint: if a single byte changes, the hash changes."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


ENVIRONMENT = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "seed": SEED,
}
print("Root:", ROOT)
print("Raw Path:", RAW_PATH)
print("Out Dir:", OUT_DIR)
ENVIRONMENT

Root: c:\Users\andre\mucsi-deusto-code\machine-learning\hotel-bookings-project
Raw Path: c:\Users\andre\mucsi-deusto-code\machine-learning\hotel-bookings-project\content\data\raw\hotels.csv
Out Dir: c:\Users\andre\mucsi-deusto-code\machine-learning\hotel-bookings-project\content\outputs\01-data-collection-and-quality


{'python': '3.14.4', 'pandas': '2.3.3', 'numpy': '2.3.5', 'seed': 42}

## 1 · Definir el dataset que necesita el proyecto

> **Pregunta:** ¿qué tiene que representar cada fila y cuándo predice el modelo?
>
> **Respuesta:** una fila es **una reserva**. El modelo predice **al registrarla**. Todo lo que ocurre
> después (cambios, llegada, cancelación) está **prohibido** como entrada.

Esta ficha se fija **antes** de mirar los datos. Si no, el dataset decide el problema en lugar de al revés.

In [41]:
DATASET_SPEC = {
    "population": "bookings from two hotels in Portugal (Resort in the Algarve, City in Lisbon) "
    "with arrival between 2015-07-01 and 2017-08-31",
    "observation_unit": "one booking",
    "candidate_key": "none (anonymized data) → we create row_id = position in the file",
    "prediction_time": "when the booking is registered (booking_date = arrival_date - lead_time)",
    "source_snapshot_time": "the day before arrival (defined by the paper)",
    "targets": {"classification": "is_canceled", "regression": "adr"},
    "intended_decision": "request a deposit and size overbooking according to cancellation risk; "
    "reference price per night",
    "forbidden_information": "everything after the booking: final status, assigned room, changes",
}
pd.Series(DATASET_SPEC, name="definition").to_frame()

,definition
population,bookings from two hotels in Portugal (Resort i...
observation_unit,one booking
candidate_key,none (anonymized data) → we create row_id = po...
prediction_time,when the booking is registered (booking_date =...
source_snapshot_time,the day before arrival (defined by the paper)
targets,"{'classification': 'is_canceled', 'regression'..."
intended_decision,request a deposit and size overbooking accordi...
forbidden_information,"everything after the booking: final status, as..."


> ⚠️ **La tensión del proyecto.** El modelo predice **al reservar**, pero cada fila es una foto de la
> **víspera de la llegada**. Cualquier columna que pueda cambiar entre esos dos momentos es sospechosa de
> fuga. La sección 4 lo comprueba.
>
> Lo dice el paper (Antonio et al., 2019): *«The data point time for each observation was defined as the
> day prior to each booking's arrival.»*

| Momento | Ejemplo | Variable |
|---|---|---|
| Reserva → **el modelo predice aquí** | 7 may 2016 | `booking_date` = `arrival_date` − `lead_time` |
| Víspera de la llegada → **foto de la fila** | 14 ago 2016 | Ninguna (lo define el paper) |
| Llegada | 15 ago 2016 | `arrival_date` |

## 2 · Cargar la fuente y conservar su contexto

> **Pregunta:** ¿el fichero es exactamente el que publicaron los autores?
>
> **Respuesta:** sí. El hash coincide y el número de filas, columnas y filas por hotel coincide con el paper.

| Campo | Valor |
|---|---|
| Dataset | *Hotel booking demand datasets*, Antonio, de Almeida y Nunes, *Data in Brief* 22 (2019) 41–49 |
| DOI · Licencia | [10.1016/j.dib.2018.11.126](https://doi.org/10.1016/j.dib.2018.11.126) · CC BY 4.0 |
| Origen | Consultas al PMS (*Property Management System*) de cada hotel |
| Copia usada | H1 (Resort) + H2 (City) unidos con la columna `hotel` (TidyTuesday 2020-02-11) |
| Momento del dato | La víspera de la llegada de cada reserva (*«the day prior to each booking's arrival»*) |
| Definición de `adr` | Suma de las transacciones de alojamiento ÷ número de noches (estándar de la American Hotel & Lodging Association) |
| Moneda de `adr` | ⚠️ **El paper no la declara.** Asumimos **euros** (hoteles en Portugal): es un supuesto, no un dato |

In [42]:
SOURCE = {
    "name": "hotel-booking-demand",
    "doi": "10.1016/j.dib.2018.11.126",
    "license": "CC BY 4.0",
    "url": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv",
    "expected_sha256": "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06",
}

if not RAW_PATH.exists():  # downloaded only if missing
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(SOURCE["url"], RAW_PATH)

raw_sha256 = sha256_file(RAW_PATH)
check(
    raw_sha256 == SOURCE["expected_sha256"],
    "Hash matches: this is the expected file.",
    "Hash mismatch: this is not the expected file.",
)

✅ Hash matches: this is the expected file.


**Leemos todo como texto** y sin convertir nada en `NaN`. Así `raw` guarda exactamente lo recibido
(`NULL`, `NA`, `Undefined`…) y ninguna interpretación queda escondida dentro de `read_csv`.

In [43]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
raw.insert(
    0, "row_id", np.arange(len(raw))
)  # technical identifier (not a business key)

source_inventory = pd.DataFrame(
    [
        {
            "source": SOURCE["name"],
            "file": RAW_PATH.name,
            "rows": len(raw),
            "columns": raw.shape[1] - 1,
            "size_mb": round(RAW_PATH.stat().st_size / 1e6, 2),
            "sha256": raw_sha256,
            "read_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
    ]
)
source_inventory.to_csv(OUT_DIR / "source_inventory.csv", index=False)
source_inventory

,source,file,rows,columns,size_mb,sha256,read_at
0,hotel-booking-demand,hotels.csv,119390,32,16.86,7c2ae42a7353905ea136e5c2287f17c92c5435826598bf...,2026-10-13T14:24:33+00:00


In [44]:
EXPECTED = {
    "rows": 119_390,
    "columns": 32,
    "Resort Hotel": 40_060,
    "City Hotel": 79_330,
}  # paper

rows_by_hotel = raw["hotel"].value_counts()

check(
    len(raw) == EXPECTED["rows"],
    "119,390 rows, as stated in the paper.",
    "Different number of rows.",
)
check(
    raw.shape[1] - 1 == EXPECTED["columns"],
    "32 columns.",
    "Different number of columns.",
)
check(
    all(rows_by_hotel[h] == EXPECTED[h] for h in ["Resort Hotel", "City Hotel"]),
    "Rows per hotel match H1 and H2.",
    "Rows per hotel differ.",
)

✅ 119,390 rows, as stated in the paper.
✅ 32 columns.
✅ Rows per hotel match H1 and H2.


### Diccionario de variables

Para cada columna: tipo analítico y **papel** en el proyecto. `outcome` = es el desenlace, no puede ser
entrada.

In [45]:
DATA_DICTIONARY = pd.DataFrame([
    ("hotel",                          "categorical", "feature",    "Resort Hotel (Algarve) or City Hotel (Lisbon)"),
    ("is_canceled",                    "binary",      "target_cls", "1 = the booking was canceled"),
    ("lead_time",                      "count_days",  "feature",    "Days between booking and arrival"),
    ("arrival_date_year",              "date_part",   "feature",    "Arrival year"),
    ("arrival_date_month",             "date_part",   "feature",    "Arrival month (English text)"),
    ("arrival_date_week_number",       "date_part",   "feature",    "Arrival week of the year"),
    ("arrival_date_day_of_month",      "date_part",   "feature",    "Arrival day of the month"),
    ("stays_in_weekend_nights",        "count",       "feature",    "Weekend nights"),
    ("stays_in_week_nights",           "count",       "feature",    "Week nights"),
    ("adults",                         "count",       "feature",    "Adults"),
    ("children",                       "count",       "feature",    "Children"),
    ("babies",                         "count",       "feature",    "Babies"),
    ("meal",                           "categorical", "feature",    "Meal plan: BB, HB, FB, SC/Undefined"),
    ("country",                        "categorical", "feature",    "Country of origin (ISO alpha-3)"),
    ("market_segment",                 "categorical", "feature",    "Market segment"),
    ("distribution_channel",           "categorical", "feature",    "Distribution channel"),
    ("is_repeated_guest",              "binary",      "feature",    "1 = repeated guest"),
    ("previous_cancellations",         "count",       "feature",    "Previous cancellations by the guest"),
    ("previous_bookings_not_canceled", "count",       "feature",    "Previous bookings not canceled"),
    ("reserved_room_type",             "categorical", "feature",    "Reserved room (code)"),
    ("assigned_room_type",             "categorical", "feature",    "Assigned room (code)"),
    ("booking_changes",                "count",       "feature",    "Changes up to arrival or cancellation"),
    ("deposit_type",                   "categorical", "feature",    "No Deposit / Non Refund / Refundable"),
    ("agent",                          "id_code",     "feature",    "Travel agency ID; NULL = not applicable"),
    ("company",                        "id_code",     "feature",    "Company ID; NULL = not applicable"),
    ("days_in_waiting_list",           "count_days",  "feature",    "Days in the waiting list"),
    ("customer_type",                  "categorical", "feature",    "Transient, Transient-Party, Contract, Group"),
    ("adr",                            "continuous",  "target_reg", "Average daily rate (price per night)"),
    ("required_car_parking_spaces",    "count",       "feature",    "Requested parking spaces"),
    ("total_of_special_requests",      "count",       "feature",    "Special requests"),
    ("reservation_status",             "categorical", "outcome",    "Canceled / Check-Out / No-Show"),
    ("reservation_status_date",        "date",        "outcome",    "Date of the last status"),
], columns=["column", "analytic_type", "role", "description"])

check(set(DATA_DICTIONARY["column"]) == set(raw.columns) - {"row_id"},
      "The dictionary covers all 32 columns.", "Dictionary and file do not match.")
DATA_DICTIONARY

✅ The dictionary covers all 32 columns.


,column,analytic_type,role,description
0,hotel,categorical,feature,Resort Hotel (Algarve) or City Hotel (Lisbon)
1,is_canceled,binary,target_cls,1 = the booking was canceled
2,lead_time,count_days,feature,Days between booking and arrival
3,arrival_date_year,date_part,feature,Arrival year
4,arrival_date_month,date_part,feature,Arrival month (English text)
5,arrival_date_week_number,date_part,feature,Arrival week of the year
6,arrival_date_day_of_month,date_part,feature,Arrival day of the month
7,stays_in_weekend_nights,count,feature,Weekend nights
8,stays_in_week_nights,count,feature,Week nights
9,adults,count,feature,Adults


**Aclaraciones del paper sobre `adr`**

| | Qué se sabe | Qué no se sabe |
|---|---|---|
| **Definición** | Precio medio **por noche**: transacciones de alojamiento ÷ noches | Si incluye impuestos o descuentos |
| **Moneda** | — | No se declara → **asumimos €** y lo anotamos en el contrato (§8) y en las limitaciones de la memoria |

### Copia tipada `df`

`raw` no se toca. En `df` convertimos a números y fechas. Son reglas fijas que **no aprenden nada** de los
datos, así que se pueden aplicar a todo el dataset sin riesgo de fuga.

In [46]:
MONTHS = {
    m: i
    for i, m in enumerate(
        [
            "January",
            "February",
            "March",
            "April",
            "May",
            "June",
            "July",
            "August",
            "September",
            "October",
            "November",
            "December",
        ],
        start=1,
    )
}

condition = DATA_DICTIONARY["analytic_type"].isin(
    ["binary", "count", "count_days", "continuous"]
)

NUMERIC = DATA_DICTIONARY.loc[condition, "column"].tolist() + [
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
]

df = raw.copy()
for col in NUMERIC:
    df[col] = pd.to_numeric(raw[col], errors="coerce")

df["arrival_date"] = pd.to_datetime(
    {
        "year": df["arrival_date_year"],
        "month": raw["arrival_date_month"].map(MONTHS),
        "day": df["arrival_date_day_of_month"],
    },
    errors="coerce",
)

df["reservation_status_date"] = pd.to_datetime(
    raw["reservation_status_date"], errors="coerce"
)

df["booking_date"] = df["arrival_date"] - pd.to_timedelta(
    df["lead_time"], unit="D"
)  # prediction time


df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Which text values could not be converted to a number?
rows = []
for col in NUMERIC:
    failed = df[col].isna()  # rows left as NaN after the conversion
    for text, n in raw.loc[failed, col].value_counts().items():
        rows.append({"column": col, "raw_value": text, "rows": n})

not_numeric = pd.DataFrame(rows, columns=["column", "raw_value", "rows"])
not_numeric

,column,raw_value,rows
0,children,NA,4


Lo único que no convierte es `NA` en `children` (4 filas): un centinela de «ausente». Se trata en el 02.

## 3 · Granularidad, clave y duplicados

> **Pregunta:** ¿qué es una fila y cómo la identificamos?
>
> **Respuesta:** una reserva, pero **sin identificador**: los autores lo eliminaron para anonimizar.
> Además, **más de una cuarta parte de las filas son copias exactas** de otra.

Sin ID no se puede saber si dos filas iguales son **la misma reserva repetida** (un error) o **dos
reservas distintas con los mismos datos** (p. ej., varias habitaciones de un grupo). Aquí solo **medimos**.
La decisión se toma en el notebook 02.

In [47]:
data_cols = [c for c in raw.columns if c != "row_id"]
dup_all = raw.duplicated(subset=data_cols, keep=False)      # every row involved
dup_extra = raw.duplicated(subset=data_cols, keep="first")  # extra row involved

duplicates = pd.Series({
    "total rows": len(raw),
    "rows involved in a duplicate": int(dup_all.sum()),
    "(%) canceled among duplicated": round(df.loc[dup_all, "is_canceled"].mean() * 100, 1),
    "(%) canceled among non-duplicated": round(df.loc[~dup_all, "is_canceled"].mean() * 100, 1),
}, name="value").to_frame()

duplicates

,value
total rows,119390.0
rows involved in a duplicate,40165.0
(%) canceled among duplicated,58.4
(%) canceled among non-duplicated,26.2


**Por qué importa:** las filas duplicadas se cancelan mucho más que el resto. Si una copia cae en *train* y
su gemela en *test*, el modelo «ha visto» la respuesta: **fuga entre particiones**. Esto condiciona cómo
se partirán los datos (notebook 04).

## 4 · Momento de corte: qué columnas existen al reservar

> **Pregunta:** ¿qué columnas no se conocen todavía cuando el modelo predice?
>
> **Respuesta:** 5 están **prohibidas** y 8 son **dudosas** (se conocen al reservar, pero pueden cambiar
> antes de la llegada).

- **sí (`yes`):** se conoce al reservar y no depende del desenlace.
- **dudoso (`uncertain`):** se conoce al reservar, pero la fila guarda su **último** valor. Se probará con y sin ella.
- **no (`no`):** aparece después de reservar o es el propio desenlace → **prohibida**.

In [48]:
AVAILABILITY = pd.DataFrame([
    ("hotel",                                  "yes",       "Fixed"),
    ("lead_time",                              "yes",       "Computed at booking time"),
    ("arrival_date_*",                         "yes",       "Date requested at booking time"),
    ("market_segment",                         "yes",       "Commercial origin"),
    ("distribution_channel",                   "yes",       "Entry channel"),
    ("is_repeated_guest",                      "yes",       "Previous history"),
    ("previous_cancellations",                 "yes",       "Previous history"),
    ("previous_bookings_not_canceled",         "yes",       "Previous history"),
    ("reserved_room_type",                     "yes",       "What is being booked"),
    ("customer_type",                          "yes",       "Contract type"),
    ("agent",                                  "yes",       "Who intermediates"),
    ("company",                                "yes",       "Who pays"),
    ("stays_in_*_nights",                      "uncertain", "May change before arrival"),
    ("adults / children / babies",             "uncertain", "May change before arrival"),
    ("meal",                                   "uncertain", "May change before arrival"),
    ("days_in_waiting_list",                   "uncertain", "Known at confirmation, not at request"),
    ("deposit_type",                           "uncertain", "Computed from payments up to arrival (paper)"),
    ("total_of_special_requests",              "uncertain", "Requests accumulate up to arrival"),
    ("country",                                "uncertain", "The paper warns: corrected at check-in"),
    ("adr (as a classification input)",        "uncertain", "May be adjusted up to arrival"),
    ("required_car_parking_spaces",            "no",        "Recorded on arrival (see evidence)"),
    ("booking_changes",                        "no",        "Counts changes up to arrival or cancellation"),
    ("assigned_room_type",                     "no",        "Assigned on arrival"),
    ("reservation_status",                     "no",        "It is the outcome"),
    ("reservation_status_date",                "no",        "Date of the outcome"),
], columns=["variable", "available_at_booking", "reason"])

AVAILABILITY.to_csv(OUT_DIR / "availability_at_prediction_time.csv", index=False)
FORBIDDEN = AVAILABILITY.query("available_at_booking == 'no'")["variable"].tolist()
UNDER_REVIEW = AVAILABILITY.query("available_at_booking == 'uncertain'")["variable"].tolist()
AVAILABILITY

,variable,available_at_booking,reason
0,hotel,yes,Fixed
1,lead_time,yes,Computed at booking time
2,arrival_date_*,yes,Date requested at booking time
3,market_segment,yes,Commercial origin
4,distribution_channel,yes,Entry channel
5,is_repeated_guest,yes,Previous history
6,previous_cancellations,yes,Previous history
7,previous_bookings_not_canceled,yes,Previous history
8,reserved_room_type,yes,What is being booked
9,customer_type,yes,Contract type


### Evidencia

**Idea:** si una columna se rellena después de reservar, las reservas canceladas (que nunca llegan) tendrán
valores muy distintos de las que llegan.

In [49]:
canc = df["is_canceled"] == 1


def pct(mask):
    return round(mask.mean() * 100, 1)


leak_evidence = pd.DataFrame(
    {
        "indicator": [
            "(%) with parking requested",
            "(%) assigned room ≠ reserved room",
            "(%) with any change",
            "(%) with country = PRT",
        ],
        "arrived": [
            pct(df.loc[~canc, "required_car_parking_spaces"] > 0),
            pct(
                raw.loc[~canc, "assigned_room_type"]
                != raw.loc[~canc, "reserved_room_type"]
            ),
            pct(df.loc[~canc, "booking_changes"] > 0),
            pct(raw.loc[~canc, "country"] == "PRT"),
        ],
        "canceled": [
            pct(df.loc[canc, "required_car_parking_spaces"] > 0),
            pct(
                raw.loc[canc, "assigned_room_type"]
                != raw.loc[canc, "reserved_room_type"]
            ),
            pct(df.loc[canc, "booking_changes"] > 0),
            pct(raw.loc[canc, "country"] == "PRT"),
        ],
    }
)
leak_evidence

,indicator,arrived,canceled
0,(%) with parking requested,9.9,0.0
1,(%) assigned room ≠ reserved room,18.8,1.8
2,(%) with any change,20.3,6.4
3,(%) with country = PRT,28.0,62.2


In [50]:
# reservation_status is is_canceled under another name
same = ((raw["reservation_status"] != "Check-Out") == canc).all()
check(same, "reservation_status matches is_canceled 100% → it is the outcome: forbidden.",
      "reservation_status does not exactly match is_canceled.")

✅ reservation_status matches is_canceled 100% → it is the outcome: forbidden.


**Lectura**

- **Parking:** **ninguna** cancelada lo tiene. Se registra al llegar → **fuga clara**.
- **Habitación asignada y cambios:** diferencias enormes, coherentes con su definición → prohibidas.
- **`country`:** la mayoría de las canceladas son PRT, muchas más que entre las que llegan. Encaja con el aviso del paper (la
  nacionalidad se corrige en el check-in) → **dudosa**: experimento con y sin ella.

## 5 · Dimensiones de calidad

> **Pregunta:** ¿cuánta calidad tienen los datos, medida con números?
>
> **Respuesta:** los objetivos están completos y son válidos. Los problemas son **duplicados**, unas
> **pocas filas imposibles** y **reservas de 0 noches**, que no sirven para la regresión.

Cada dimensión responde a una pregunta distinta:

| Dimensión | Pregunta |
|---|---|
| Completitud | ¿Falta el dato? (aquí la ausencia viene como texto: `NULL`, `NA`) |
| Unicidad | ¿Hay filas repetidas? |
| Validez | ¿El valor está dentro de lo posible? |
| Consistencia | ¿Las columnas cuadran entre sí? |
| Puntualidad | ¿El dato llega a tiempo? (no aplica: es un *snapshot* publicado en 2019) |

In [51]:
ADR_MAX = 1_000                                            # domain limit (the 2nd highest value is 510)
guests = df["adults"] + df["children"].fillna(0) + df["babies"]
jan1 = pd.to_datetime(dict(year=df["arrival_date_year"], month=1, day=1))
expected_week = (df["arrival_date"].dt.dayofyear - 1 + (jan1.dt.weekday + 1) % 7) // 7 + 1  # weeks starting on Sunday
checkout = raw["reservation_status"] == "Check-Out"
checkout_ok = df["reservation_status_date"] == df["arrival_date"] + pd.to_timedelta(df["total_nights"], unit="D")

quality_metrics = pd.DataFrame([
    # dimension, metric, value, unit
    ("completeness", "is_canceled present",                pct(df["is_canceled"].notna()),         "%"),
    ("completeness", "adr present",                        pct(df["adr"].notna()),                 "%"),
    ("completeness", "country present (≠ NULL)",           pct(raw["country"] != "NULL"),          "%"),
    ("completeness", "children present (≠ NA)",            pct(raw["children"] != "NA"),           "%"),
    ("uniqueness",   "surplus exact copies",               pct(dup_extra),                         "%"),
    ("validity",     "is_canceled in {0, 1}",              pct(df["is_canceled"].isin([0, 1])),    "%"),
    ("validity",     "valid arrival date",                 pct(df["arrival_date"].notna()),        "%"),
    ("validity",     "rows with adr < 0",                  int((df["adr"] < 0).sum()),             "rows"),
    ("validity",     f"rows with adr > {ADR_MAX}",         int((df["adr"] > ADR_MAX).sum()),       "rows"),
    ("validity",     "rows without guests",                int((guests == 0).sum()),               "rows"),
    ("validity",     "rows with 0 nights",                 int((df["total_nights"] == 0).sum()),   "rows"),
    ("validity",     "non ISO alpha-3 country code",       int((~raw["country"].str.fullmatch(r"[A-Z]{3}")
                                                                & raw["country"].ne("NULL")).sum()), "rows"),
    ("consistency",  "week of year consistent with the date", pct(expected_week == df["arrival_date_week_number"]), "%"),
    ("consistency",  "Check-Out: status date = arrival + nights", pct(checkout_ok[checkout]),      "%"),
], columns=["dimension", "metric", "value", "unit"])

quality_metrics.to_csv(OUT_DIR / "quality_metrics.csv", index=False)
quality_metrics

,dimension,metric,value,unit
0,completeness,is_canceled present,100.0,%
1,completeness,adr present,100.0,%
2,completeness,country present (≠ NULL),99.6,%
3,completeness,children present (≠ NA),100.0,%
4,uniqueness,surplus exact copies,26.8,%
5,validity,"is_canceled in {0, 1}",100.0,%
6,validity,valid arrival date,100.0,%
7,validity,rows with adr < 0,1.0,rows
8,validity,rows with adr > 1000,1.0,rows
9,validity,rows without guests,180.0,rows


**Lectura**

- **Filas imposibles** (1 `adr` negativo, 1 `adr` de 5 400 y 180 sin huéspedes) → irán a **cuarentena** en el 02.
- **715 reservas de 0 noches**: `adr` es «por noche», así que sin noches no está definido. Valen para la
  clasificación, **no para la regresión**.
- **Código de país no ISO:** `CN` aparece junto a `CHN`. Se decide en el 02 si se unifican.
- **Consistencia ≈ 100 %:** las fechas cuadran. La semana del año empieza en **domingo**, no sigue la norma ISO.

## 6 · Cobertura y representatividad

> **Pregunta:** ¿a qué realidad representan estos datos?
>
> **Respuesta:** a **dos hoteles portugueses entre 2015 y 2017**, con clientes mayoritariamente de
> Portugal. El modelo **no puede presentarse como general** para cualquier hotel o época.

- **Cobertura:** ¿tenemos todo lo que dice la fuente?
- **Representatividad:** ¿cómo se reparte ese volumen?

In [52]:
period_ok = df["arrival_date"].between("2015-07-01", "2017-08-31")
check(period_ok.all(), "All arrivals fall within the declared period.", "There are arrivals outside the period.")

# How many times does each month appear? (26 months: July and August appear 3 times)
months_by_year = pd.crosstab(df["arrival_date_year"], df["arrival_date"].dt.month).gt(0).astype(int)
months_by_year.loc["times"] = months_by_year.sum()
months_by_year

✅ All arrivals fall within the declared period.


arrival_date,1,2,3,4,5,6,7,8,9,10,11,12
arrival_date_year,,,,,,,,,,,,
2015,0,0,0,0,0,0,1,1,1,1,1,1
2016,1,1,1,1,1,1,1,1,1,1,1,1
2017,1,1,1,1,1,1,1,1,0,0,0,0
times,2,2,2,2,2,2,3,3,2,2,2,2


In [53]:
country_share = raw["country"].value_counts(normalize=True) * 100
representativity = pd.Series({
    "% City Hotel": pct((raw["hotel"] == "City Hotel")),
    "distinct countries": raw["country"].nunique(),
    "% Portugal (PRT)": round(country_share["PRT"], 1),
    "% top-5 countries": round(country_share.head(5).sum(), 1),
    "countries with < 10 bookings": int((raw["country"].value_counts() < 10).sum()),
}, name="value").to_frame()
representativity

,value
% City Hotel,66.4
distinct countries,178.0
% Portugal (PRT),40.7
% top-5 countries,72.9
countries with < 10 bookings,81.0


**Lectura**

- **Periodo incompleto:** julio y agosto aparecen tres veces y el resto de meses, dos. Cualquier media por
  mes está sesgada hacia el verano. Una partición **temporal** (entrenar con el pasado y evaluar con lo más
  reciente) es la que imita el uso real.
- **Origen concentrado:** Portugal es el 41 % y hay decenas de países con muy pocas reservas → habrá que
  agruparlos en «otros» (notebook 03).

## 7 · Scorecard: ¿puede avanzar el lote?

> **Pregunta:** con todo lo medido, ¿pasamos al notebook 02?
>
> **Respuesta:** **sí, con condiciones** (`CONTINUE_WITH_CONDITIONS`). Ningún fallo bloquea el dataset
> entero. Los fallos afectan a filas concretas o a columnas concretas.

Cada control convierte una métrica en una decisión: **métrica → regla (umbral) → severidad → acción**.
No se hace una media: **un solo fallo crítico del dataset bloquea el lote**.

La columna `scope` indica a qué afecta cada fallo:

- `dataset` → al lote entero. Si falla y es crítico, **bloquea**.
- `row` → a filas concretas, que van a **cuarentena** en el 02.
- `features` → a qué columnas pueden entrar en el modelo.

In [54]:
qm = quality_metrics.set_index("metric")["value"]

controls = [
    # control, dimension, scope, observed, operator, threshold, severity, action on failure
    ("source_hash",            "integrity",    "dataset",  float(raw_sha256 == SOURCE["expected_sha256"]), "==", 1,                    "critical", "stop and download again"),
    ("rows",                   "integrity",    "dataset",  len(raw),                                       "==", EXPECTED["rows"],     "critical", "stop and investigate"),
    ("columns",                "schema",       "dataset",  raw.shape[1] - 1,                               "==", EXPECTED["columns"],  "critical", "stop and investigate"),
    ("is_canceled_valid_%",    "validity",     "dataset",  qm["is_canceled in {0, 1}"],                    "==", 100,                  "critical", "stop and investigate"),
    ("adr_present_%",          "completeness", "dataset",  qm["adr present"],                              "==", 100,                  "critical", "stop and investigate"),
    ("arrival_date_valid_%",   "validity",     "dataset",  qm["valid arrival date"],                       "==", 100,                  "critical", "stop and investigate"),
    ("period_covered_%",       "coverage",     "dataset",  pct(period_ok),                                 "==", 100,                  "critical", "stop and investigate"),
    ("exact_copies_%",         "uniqueness",   "dataset",  qm["surplus exact copies"],                     "<",  5,                    "warning",  "define a duplicates policy"),
    ("country_present_%",      "completeness", "dataset",  qm["country present (≠ NULL)"],                 ">=", 99,                   "warning",  "flag missingness"),
    ("negative_adr",           "validity",     "row",      qm["rows with adr < 0"],                        "==", 0,                    "critical", "quarantine"),
    ("adr_out_of_domain",      "validity",     "row",      qm[f"rows with adr > {ADR_MAX}"],               "==", 0,                    "critical", "quarantine"),
    ("no_guests",              "validity",     "row",      qm["rows without guests"],                      "==", 0,                    "critical", "quarantine"),
    ("zero_nights",            "validity",     "row",      qm["rows with 0 nights"],                       "==", 0,                    "warning",  "exclude from the regression"),
    ("non_iso_country",        "validity",     "row",      qm["non ISO alpha-3 country code"],             "==", 0,                    "warning",  "review CN/CHN mapping"),
    ("forbidden_columns",      "leakage",      "features", len(FORBIDDEN),                                 "==", 0,                    "critical", "exclude them from the features"),
]
scorecard = pd.DataFrame(controls, columns=["control", "dimension", "scope", "observed",
                                            "operator", "threshold", "severity", "on_failure"])

In [55]:
OPS = {
    "==": lambda o, t: np.isclose(o, t),
    "<": lambda o, t: o < t,
    ">=": lambda o, t: o >= t,
}

scorecard["status"] = [
    "PASS" if OPS[op](o, t) else "FAIL"
    for o, op, t in zip(
        scorecard["observed"], scorecard["operator"], scorecard["threshold"]
    )
]
scorecard.to_csv(OUT_DIR / "scorecard.csv", index=False)
scorecard

,control,dimension,scope,observed,operator,threshold,severity,on_failure,status
0,source_hash,integrity,dataset,1.0,==,1,critical,stop and download again,PASS
1,rows,integrity,dataset,119390.0,==,119390,critical,stop and investigate,PASS
2,columns,schema,dataset,32.0,==,32,critical,stop and investigate,PASS
3,is_canceled_valid_%,validity,dataset,100.0,==,100,critical,stop and investigate,PASS
4,adr_present_%,completeness,dataset,100.0,==,100,critical,stop and investigate,PASS
5,arrival_date_valid_%,validity,dataset,100.0,==,100,critical,stop and investigate,PASS
6,period_covered_%,coverage,dataset,100.0,==,100,critical,stop and investigate,PASS
7,exact_copies_%,uniqueness,dataset,26.8,<,5,warning,define a duplicates policy,FAIL
8,country_present_%,completeness,dataset,99.6,>=,99,warning,flag missingness,PASS
9,negative_adr,validity,row,1.0,==,0,critical,quarantine,FAIL


### Política del lote

| Situación | Decisión |
|---|---|
| Falla un control **crítico** de `dataset` | `QUARANTINE`: el lote no avanza |
| Falla cualquier otro control | `CONTINUE_WITH_CONDITIONS`: avanza con una lista de condiciones |
| Todo pasa | `PUBLISH` |

`forbidden_columns` falla **a propósito**: las columnas existen en el fichero. Se cumple cuando el
notebook 03 las deje fuera de las *features*.

In [56]:
failed = scorecard[scorecard["status"] == "FAIL"]
blocking = failed[(failed["severity"] == "critical") & (failed["scope"] == "dataset")]

batch_decision = "QUARANTINE" if len(blocking) else ("CONTINUE_WITH_CONDITIONS" if len(failed) else "PUBLISH")
print("Batch decision:", batch_decision)
failed[["control", "scope", "observed", "severity", "on_failure"]]

Batch decision: CONTINUE_WITH_CONDITIONS


,control,scope,observed,severity,on_failure
7,exact_copies_%,dataset,26.8,warning,define a duplicates policy
9,negative_adr,row,1.0,critical,quarantine
10,adr_out_of_domain,row,1.0,critical,quarantine
11,no_guests,row,180.0,critical,quarantine
12,zero_nights,row,715.0,warning,exclude from the regression
13,non_iso_country,row,1279.0,warning,review CN/CHN mapping
14,forbidden_columns,features,5.0,critical,exclude them from the features


## 8 · Contrato de datos

> **Pregunta:** ¿qué le garantizamos al notebook 02?
>
> **Respuesta:** un JSON con los seis bloques del Lab 01. Si algo de esto cambia (fichero, columnas o
> significado de un valor), el contrato sube de versión.

| Bloque | Qué dice |
|---|---|
| identidad | qué dataset es, versión, hash |
| propósito | para qué se usa, momento de predicción, columnas prohibidas |
| esquema | tipo y papel de cada columna, y qué significan los centinelas |
| calidad | los controles del scorecard |
| servicio | cómo se obtiene y con qué frecuencia se actualiza |
| evolución | cuándo cambia de versión |

In [57]:
CONTRACT_VERSION = "1.0.0"
SENTINELS = {   # text values with a special meaning (paper)
    "children": {"value": "NA", "meaning": "missing"},
    "country": {"value": "NULL", "meaning": "missing"},
    "agent": {"value": "NULL", "meaning": "not applicable (booking without an agency)"},
    "company": {"value": "NULL", "meaning": "not applicable (booking without a company)"},
    "meal": {"value": "Undefined", "meaning": "same as SC (no meal plan)"},
}

schema = {r.column: {"type": r.analytic_type, "role": r.role, **({"sentinel": SENTINELS[r.column]}
          if r.column in SENTINELS else {})} for r in DATA_DICTIONARY.itertuples()}
schema["adr"].update({"min": 0, "max": ADR_MAX, "unit": "€/night (currency not stated in the paper)"})

data_contract = {
    "identity":  {"dataset": SOURCE["name"], "version": CONTRACT_VERSION, "doi": SOURCE["doi"],
                  "license": SOURCE["license"], "sha256": raw_sha256},
    "purpose":   {"targets": DATASET_SPEC["targets"], "unit": DATASET_SPEC["observation_unit"],
                  "prediction_time": DATASET_SPEC["prediction_time"],
                  "forbidden_features": FORBIDDEN, "features_under_review": UNDER_REVIEW},
    "schema":    schema,
    "quality":   scorecard[["control", "operator", "threshold", "severity", "on_failure"]].to_dict("records"),
    "service":   {"type": "static snapshot", "refresh": "none", "access": SOURCE["url"],
                  "period": "arrivals 2015-07-01 to 2017-08-31"},
    "evolution": {"policy": "change of file, columns, thresholds or meaning of a sentinel → new version"},
}
(OUT_DIR / "data_contract.json").write_text(json.dumps(data_contract, indent=2, ensure_ascii=False), encoding="utf-8")
print("Contract saved with", len(schema), "columns in the schema.")

Contract saved with 32 columns in the schema.


## 9 · Evidencia reproducible

> **Pregunta:** ¿cómo demuestra otra persona que obtiene exactamente esto?
>
> **Respuesta:** con dos archivos.
>
> - El **manifiesto** une fuente, hash, entorno, contrato y decisión.
> - El **resumen de calidad** es lo primero que lee el notebook 02.

In [58]:
quality_summary = {
    "dataset": SOURCE["name"],
    "contract_version": CONTRACT_VERSION,
    "decision": batch_decision,
    "passed_controls": int((scorecard["status"] == "PASS").sum()),
    "failed_controls": int((scorecard["status"] == "FAIL").sum()),
    "rows_to_quarantine": int(((df["adr"] < 0) | (df["adr"] > ADR_MAX) | (guests == 0)).sum()),
    "rows_excluded_from_regression": int((df["total_nights"] == 0).sum()),
    "forbidden_features": FORBIDDEN,
    "features_under_review": UNDER_REVIEW,
    "pending_decisions_for_02": [
        "exact duplicates: drop, group within the split or weight",
        "adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not",
        "CN versus CHN: unify or not",
        "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC",
    ],
}
(OUT_DIR / "quality_summary.json").write_text(json.dumps(quality_summary, indent=2, ensure_ascii=False), encoding="utf-8")

run_manifest = {
    "notebook": "01-data-collection-and-quality",
    "executed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment": ENVIRONMENT,
    "source": {"path": RAW_PATH.relative_to(ROOT).as_posix(), "url": SOURCE["url"], "sha256": raw_sha256,
               "rows": len(raw), "columns": raw.shape[1] - 1},
    "contract_version": CONTRACT_VERSION,
    "decision": batch_decision,
    "failed_controls": failed["control"].tolist(),
}
run_manifest["artifacts"] = sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()) + ["run_manifest.json"]
(OUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("Files generated in", OUT_DIR.relative_to(ROOT).as_posix())
for name in sorted(set(run_manifest["artifacts"])):
    print("  •", name)
quality_summary

Files generated in content/outputs/01-data-collection-and-quality
  • availability_at_prediction_time.csv
  • data_contract.json
  • quality_metrics.csv
  • quality_summary.json
  • run_manifest.json
  • scorecard.csv
  • source_inventory.csv


{'dataset': 'hotel-booking-demand',
 'contract_version': '1.0.0',
 'decision': 'CONTINUE_WITH_CONDITIONS',
 'passed_controls': 8,
 'failed_controls': 7,
 'rows_to_quarantine': 182,
 'rows_excluded_from_regression': 715,
 'forbidden_features': ['required_car_parking_spaces',
  'booking_changes',
  'assigned_room_type',
  'reservation_status',
  'reservation_status_date'],
 'features_under_review': ['stays_in_*_nights',
  'adults / children / babies',
  'meal',
  'days_in_waiting_list',
  'deposit_type',
  'total_of_special_requests',
  'country',
  'adr (as a classification input)'],
 'pending_decisions_for_02': ['exact duplicates: drop, group within the split or weight',
  'adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not',
  'CN versus CHN: unify or not',
  "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC"]}

## TL;DR final: qué pasa al notebook 02

| | |
|---|---|
| **Decisión** | `CONTINUE_WITH_CONDITIONS`: el dataset es el correcto y está completo |
| **Prohibidas** | `reservation_status`, `reservation_status_date`, `assigned_room_type`, `booking_changes`, `required_car_parking_spaces` |
| **Dudosas** | `country`, `deposit_type`, `total_of_special_requests`, noches, huéspedes, `meal`, `days_in_waiting_list` y `adr` como entrada de la clasificación → experimento con y sin ellas |
| **Cuarentena** | 182 filas imposibles (`adr` < 0, `adr` > 1 000, sin huéspedes) |
| **Fuera de la regresión** | 715 reservas de 0 noches |
| **Riesgo principal** | 26,8 % de copias exactas → fuga entre *train* y *test* si no se controla |
| **Limitación** | dos hoteles, un país, 2015–2017 → validez externa limitada |

**Siguiente:** `02-eda-and-cleaning.ipynb`. Explora a fondo (perfiles, ausencias por segmento, objetivos y
gráficos), aplica las decisiones pendientes y produce el dataset limpio.